In [ ]:
!pip install langchain-text-splitters

In [ ]:
!pip install bert-score

In [1]:
import json
import re
from pathlib import Path

import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

INPUT_MD = "output.md"
INPUT_MD_2 = "output2.md"
OUTPUT_JSON = "chunks.json"
CHUNK_OVERLAP = 153
CHUNK_SIZE = 1024 - CHUNK_OVERLAP
MIN_CHUNK_SIZE = 512 - CHUNK_OVERLAP
TOKENIZER = "cl100k_base"

RE_IMG = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")
RE_BLOCK_FORMULA = re.compile(r"\$\$(.+?)\$\$", re.DOTALL)
RE_INLINE_FORMULA = re.compile(r"(?<!\$)\$([^\$\n]+?)\$(?!\$)")
RE_HTML_TABLE = re.compile(r"<table\b.*?</table>", re.DOTALL | re.I)
RE_MD_TABLE = re.compile(r"(?:^\|[^\n]*\|\s*$\n?){2,}", re.M)
PH_RE = re.compile(r"(\[\[(?:FORMULA|TABLE|IMAGE)_\d+\]\])")
RE_SENT = re.compile(r"(?<=[.!?])\s+")
RE_DATE = re.compile(r"\b(?:\d{1,2}[./-]\d{1,2}[./-]\d{4}|\d{4}[./-]\d{1,2}[./-]\d{1,2})\b")
RE_NUMBER = re.compile(r"""(?<![\w.])[+-]?(?:\d{1,3}(?:[ _]\d{3})+|\d+)(?:[.,]\d+)?(?:[eE][+-]?\d+)?(?![\w.])""",re.VERBOSE)

enc = tiktoken.get_encoding(TOKENIZER)

FORMULAS = {}
TABLES = {}
IMAGES = {}
DATES = {}
NUMBERS={}
STORE = {}

def protect_pattern(text, pattern, store, prefix, block=False):
    def repl(m):
        key = f"[[{prefix}_{len(store):04d}]]"
        store[key] = m.group(0)
        return f"\n\n{key}\n\n" if block else key
    return pattern.sub(repl, text)


def protect_images(text):
    def repl(m):
        alt, src = m.group(1), m.group(2)
        key = f"[[IMAGE_{len(IMAGES):04d}]]"
        IMAGES[key] = {
            "alt": alt,
            "src": None if src.startswith("data:image") else src,
            "is_base64": src.startswith("data:image"),
            "src_len": len(src),
        }
        return key
    return RE_IMG.sub(repl, text)


def protect(text):
    text = protect_pattern(text, RE_BLOCK_FORMULA, FORMULAS, "FORMULA", block=True)
    text = protect_pattern(text, RE_HTML_TABLE, TABLES, "TABLE", block=True)
    text = protect_pattern(text, RE_MD_TABLE, TABLES, "TABLE", block=True)
    text = protect_pattern(text, RE_INLINE_FORMULA, FORMULAS, "FORMULA", block=False)
    text = protect_pattern(text, RE_DATE, DATES, "DATE", block=True)
    text = protect_pattern(text, RE_NUMBER, NUMBERS, "NUMBER", block=True)
    return text


def restore(text):
    for k, v in STORE.items():
        if k in text:
            text = text.replace(k, v)
    return text


def count(text):
    total = 0
    for part in PH_RE.split(text):
        if not part:
            continue
        value = STORE.get(part)
        total += len(enc.encode(value if isinstance(value, str) else part))
    return total

def _tail_for_overlap(text, max_tokens):
    sentences = RE_SENT.split(text)
    picked, total = [], 0
    for s in reversed(sentences):
        if PH_RE.search(s):
            continue
        s_tokens = count(s)
        if s_tokens > max_tokens:
            continue
        if picked and total + s_tokens > max_tokens:
            break
        picked.insert(0, s)
        total += s_tokens
        if total >= max_tokens:
            break
    return " ".join(picked)

def add_overlap(chunks, overlap_tokens, chunk_size):
    if len(chunks) < 2:
        return chunks
    result = [chunks[0]]
    for i in range(1, len(chunks)):
        budget = chunk_size - count(chunks[i])
        budget = max(0, min(budget, overlap_tokens))
        tail = _tail_for_overlap(chunks[i-1], budget) if budget > 0 else ""
        result.append(f"{tail}\n\n{chunks[i]}" if tail else chunks[i])
    return result

def merge_small(chunks):
    if not chunks:
        return []
    merged, buf = [], ""
    for c in chunks:
        cand = f"{buf}\n\n{c}".strip() if buf else c
        if count(cand) <= CHUNK_SIZE:
            buf = cand
        else:
            if buf:
                merged.append(buf)
            buf = c
    if buf:
        merged.append(buf)

    i = 0
    while i < len(merged):
        if count(merged[i]) >= MIN_CHUNK_SIZE:
            i += 1
            continue
        done = False
        if i > 0 and count(merged[i - 1] + "\n\n" + merged[i]) <= CHUNK_SIZE:
            merged[i - 1] += "\n\n" + merged[i]
            merged.pop(i)
            done = True
        if not done and i < len(merged) - 1 and count(merged[i] + "\n\n" + merged[i + 1]) <= CHUNK_SIZE:
            merged[i + 1] = merged[i] + "\n\n" + merged[i + 1]
            merged.pop(i)
            done = True
        if not done:
            i += 1
    return merged


def protect_text(text):
    FORMULAS.clear()
    TABLES.clear()
    IMAGES.clear()
    STORE.clear()
    NUMBERS.clear()
    DATES.clear()

    text = protect_images(text)
    protected = protect(text)

    STORE.update(FORMULAS)
    STORE.update(TABLES)
    STORE.update(DATES)
    STORE.update(NUMBERS)
    return protected
    # splitter = RecursiveCharacterTextSplitter(
    #     chunk_size=CHUNK_SIZE,
    #     chunk_overlap=0,
    #     length_function=count,
    #     separators=["\n\n", "\n", " ", ""],
    #     keep_separator=True,
    # )
    # pieces = splitter.split_text(protected)
    # pieces = merge_small(pieces)
    # pieces = add_overlap(pieces, CHUNK_OVERLAP, CHUNK_SIZE)

    # chunks = []
    # for i, p in enumerate(pieces):
    #     restored = restore(p)
    #     img_refs = [{"placeholder": k, **IMAGES[k]} for k in IMAGES if k in p]
    #     chunks.append({
    #         "text": restored,
    #         "masked_text": p,
    #         "metadata": {
    #             "index": i,
    #             "tokens": len(enc.encode(restored)),
    #             "images": img_refs,
    #             "has_formula": any(k in p for k in FORMULAS),
    #             "has_table": any(k in p for k in TABLES),
    #             "has_image": bool(img_refs),
    #             "n_atomic": len(PH_RE.findall(p)),
    #         },
    #     })
    #return chunks

In [2]:
text1 = Path(INPUT_MD).read_text(encoding="utf-8")
text2 = Path(INPUT_MD_2).read_text(encoding="utf-8")


In [4]:
protected = protect_text(text1+text2)

# Path(OUTPUT_JSON).write_text(
#     json.dumps(chunks, ensure_ascii=False, indent=2),
#     encoding="utf-8",
# )

In [5]:
STORE

{'[[FORMULA_0000]]': '$$\nE = 2 \\lim_{\\Delta l \\to 0} \\sum_{0}^{l} A_{\\mathrm{yx}} \\cdot V_{\\mathrm{zx},\\mathrm{zx}} = \\frac{2}{t} \\cdot \\int_{0}^{l} A_{\\mathrm{yx}}(l) \\cdot S(l)   dl \\cdot \\left[ \\frac{\\Delta l}{\\mathrm{CM}^2} \\right].\n$$',
 '[[FORMULA_0001]]': '$\\mathrm{OT}+20\\mathrm{~\\pi}0-196^{\\mathrm{~0}}\\mathrm{C}$',
 '[[FORMULA_0002]]': '$4 5 0 \\times 3 3 0 \\times 2 7 { , } 7$',
 '[[FORMULA_0003]]': '$5 5 0 \\times 3 3 0 \\times 2 3$',
 '[[FORMULA_0004]]': '$1 9 6 ~ ^ { \\circ } \\mathrm { C } _ { 3 }$',
 '[[FORMULA_0005]]': '$^ { 0 , 2 }$',
 '[[FORMULA_0006]]': '$( \\mathrm{M} \\bot \\mathrm{sw/M}^2 )$',
 '[[FORMULA_0007]]': '$( \\Gamma _ { \\mathrm { {   H C I } } } )$',
 '[[FORMULA_0008]]': '$( \\times ~ 5 ~ \\textbar { - } ~ 1 0 )$',
 '[[FORMULA_0009]]': '$8 0 ~ ^ { \\circ } \\mathrm { C }$',
 '[[FORMULA_0010]]': '$\\left( \\Delta = \\mathrm{KCU_{max}} - \\mathrm{KCU_{min}} \\right)$',
 '[[FORMULA_0011]]': '$175\\  五  \\times /  cm ^2$',
 '[[FORMU

In [ ]:
# sizes = [c["metadata"]["tokens"] for c in chunks]
# big = [c for c in chunks if c["metadata"]["tokens"] > 1024]
# print(f"Кол-во чанков: {len(chunks)}")
# print(f"Размеры: min={min(sizes)}, avg={sum(sizes)//len(sizes)}, max={max(sizes)}")
# print(f"Кол-во формул: {len(FORMULAS)} | Кол-во таблиц: {len(TABLES)} | Кол-во картинок: {len(IMAGES)}")
# print(f"Чанков с формулой: {sum(c['metadata']['has_formula'] for c in chunks)}, "
#       f"с таблицей: {sum(c['metadata']['has_table'] for c in chunks)}, "
#       f"с картинкой: {sum(c['metadata']['has_image'] for c in chunks)}")
# print(f"Чанков, превышающих лимит ({1024}): {len(big)}")

## Оценка разбиения GraphRAG text_units

In [ ]:
GRAPHRAG_CHUNKS_DIR = "graphrag_preprocessed_rubert"
GRAPHRAG_HIT_RATE_TOP_K = 10
GRAPHRAG_STD_TOKENS_LIMIT = 200
GRAPHRAG_BROKEN_SENTENCES_LIMIT = 5.0
GRAPHRAG_BROKEN_PARAGRAPHS_LIMIT = 2.0
GRAPHRAG_BROKEN_ATOMIC_TARGET = 0.0
GRAPHRAG_HIT_RATE_LIMIT = 90.0


def status_max(value, limit):
    return "OK" if value <= limit else "FAIL"


def status_min(value, limit):
    return "OK" if value >= limit else "FAIL"


def resolve_graphrag_output_dir(path):
    candidates = [
        Path(path),
        Path("..") / path,
        Path("..") / ".." / path,
    ]
    for candidate in candidates:
        if (candidate / "output" / "text_units.parquet").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Не найдена папка GraphRAG с text_units.parquet: {path}")


def read_graphrag_text_units(graph_dir):
    text_units_path = graph_dir / "output" / "text_units.parquet"
    try:
        text_units = pd.read_parquet(text_units_path)
        engine = "pyarrow"
    except Exception:
        text_units = pd.read_parquet(text_units_path, engine="fastparquet")
        engine = "fastparquet"

    if "text" not in text_units.columns:
        raise ValueError("В text_units.parquet нет колонки 'text'.")
    return text_units, engine


def read_graphrag_source_text(graph_dir):
    input_dir = graph_dir / "input"
    paths = sorted(input_dir.glob("*.md")) + sorted(input_dir.glob("*.txt"))
    if not paths:
        raise FileNotFoundError(f"В {input_dir} не найдены исходные .md/.txt файлы.")
    return "\n\n".join(path.read_text(encoding="utf-8", errors="ignore") for path in paths)


def norm_for_chunk_eval(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def text_unit_token_sizes(text_units):
    if "n_tokens" in text_units.columns:
        sizes = text_units["n_tokens"].fillna(0).astype(int).tolist()
        if any(size > 0 for size in sizes):
            return sizes
    return [len(enc.encode(str(text))) for text in text_units["text"].fillna("")]


def extract_source_sentences(source_text, min_chars=25):
    clean_text = re.sub(r"!\[[^\]]*\]\([^)]+\)", " ", source_text)
    clean_text = re.sub(r"<table\b.*?</table>", " ", clean_text, flags=re.DOTALL | re.I)
    sentences = []
    for block in re.split(r"\n\s*\n+", clean_text):
        block = norm_for_chunk_eval(block)
        if not block or block.startswith("|"):
            continue
        for sentence in re.split(r"(?<=[.!?\u2026])\s+", block):
            sentence = norm_for_chunk_eval(sentence)
            if len(sentence) >= min_chars and re.search(r"[.!?\u2026][\"')\]\}»”]*$", sentence):
                sentences.append(sentence)
    return sentences


def extract_source_paragraphs(source_text, max_tokens=None, min_chars=40):
    paragraphs = []
    for block in re.split(r"\n\s*\n+", source_text.replace("\r\n", "\n")):
        paragraph = norm_for_chunk_eval(block)
        if len(paragraph) < min_chars:
            continue
        if re.match(r"^\s*(<table\b|</table>|!\[|\|)", block, re.I):
            continue
        if max_tokens is not None and len(enc.encode(paragraph)) > max_tokens:
            continue
        paragraphs.append(paragraph)
    return paragraphs


def broken_items(items, chunk_texts):
    normalized_chunks = [norm_for_chunk_eval(chunk) for chunk in chunk_texts]
    broken = [
        item
        for item in items
        if not any(norm_for_chunk_eval(item) in chunk for chunk in normalized_chunks)
    ]
    pct = len(broken) / max(1, len(items)) * 100
    return {"total": len(items), "broken": len(broken), "pct": pct, "examples": broken[:5]}


def formula_table_integrity(source_text, chunk_texts):
    formula_re = re.compile(r"\$\$.*?\$\$|(?<!\$)\$[^\$\n]+?\$(?!\$)", re.DOTALL)
    table_re = re.compile(r"<table\b.*?</table>|(?:^\|[^\n]*\|\s*$\n?){2,}", re.I | re.M | re.DOTALL)
    formulas = [item for item in formula_re.findall(source_text) if len(norm_for_chunk_eval(item)) > 3]
    tables = [item for item in table_re.findall(source_text) if len(norm_for_chunk_eval(item)) > 10]
    return {
        "formulas": broken_items(formulas, chunk_texts),
        "tables": broken_items(tables, chunk_texts),
    }


def graphrag_eval_queries(source_text):
    queries = [
        ("В чем состоит цель работы Скородумова?", "Цель работы"),
        ("Где выполнена работа Скородумова?", "МИСиС"),
        ("Какая ведущая организация указана у Скородумова?", "Выксунский металлургический завод"),
        ("Какие стали использовались как объект исследования?", "К52"),
        ("Какой метод использован для обработки кривых ударной вязкости?", "максимума правдоподобия"),
        ("Какой параметр излома наиболее информативен для К65?", "уширение"),
        ("В чем состоит цель работы Струина?", "Цель работы"),
        ("Какие классы прочности труб исследуются?", "К60, К65"),
        ("Какой газопровод связан с трубами К65?", "Бованенково-Ухта"),
        ("Какая зона сварного соединения является зоной локальной хрупкости?", "участок крупного зерна"),
        ("Какие документы Газпрома включили критерии?", "СТО Газпром"),
        ("Где выполнена работа Струина?", "РосНИТИ"),
    ]
    return [
        {"query": query, "answer_pattern": answer}
        for query, answer in queries
        if answer.lower() in source_text.lower()
    ]


def hit_rate_for_text_units(chunk_texts, eval_queries, top_k=GRAPHRAG_HIT_RATE_TOP_K):
    if not chunk_texts or not eval_queries:
        return {"hits": 0, "total": 0, "pct": 0.0, "misses": []}

    vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=True)
    chunk_matrix = vectorizer.fit_transform(chunk_texts)
    query_matrix = vectorizer.transform([item["query"] for item in eval_queries])
    similarities = cosine_similarity(query_matrix, chunk_matrix)

    hits = 0
    misses = []
    for row, item in enumerate(eval_queries):
        top_indices = np.argsort(similarities[row])[::-1][:top_k]
        found = any(item["answer_pattern"].lower() in chunk_texts[index].lower() for index in top_indices)
        hits += int(found)
        if not found:
            misses.append({
                "query": item["query"],
                "answer_pattern": item["answer_pattern"],
                "top_indices": top_indices.tolist(),
            })

    return {
        "hits": hits,
        "total": len(eval_queries),
        "pct": hits / max(1, len(eval_queries)) * 100,
        "misses": misses,
    }


def evaluate_graphrag_text_units(graph_dir_path=GRAPHRAG_CHUNKS_DIR):
    graph_dir = resolve_graphrag_output_dir(graph_dir_path)
    source_text = read_graphrag_source_text(graph_dir)
    text_units, parquet_engine = read_graphrag_text_units(graph_dir)

    chunk_texts = text_units["text"].fillna("").astype(str).tolist()
    token_sizes = text_unit_token_sizes(text_units)
    max_chunk_tokens = max(token_sizes) if token_sizes else None

    sentences = extract_source_sentences(source_text)
    paragraphs = extract_source_paragraphs(source_text, max_tokens=max_chunk_tokens)
    sentence_report = broken_items(sentences, chunk_texts)
    paragraph_report = broken_items(paragraphs, chunk_texts)
    atomic_report = formula_table_integrity(source_text, chunk_texts)
    eval_queries = graphrag_eval_queries(source_text)
    hit_report_at_3 = hit_rate_for_text_units(chunk_texts, eval_queries, top_k=3)
    hit_report = hit_rate_for_text_units(chunk_texts, eval_queries, top_k=GRAPHRAG_HIT_RATE_TOP_K)

    report = {
        "graph_dir": str(graph_dir),
        "parquet_engine": parquet_engine,
        "chunks_total": len(chunk_texts),
        "mean_size_tokens": float(np.mean(token_sizes)) if token_sizes else 0.0,
        "std_size_tokens": float(np.std(token_sizes, ddof=1)) if len(token_sizes) > 1 else 0.0,
        "min_size_tokens": min(token_sizes) if token_sizes else 0,
        "max_size_tokens": max(token_sizes) if token_sizes else 0,
        "broken_sentences": sentence_report,
        "broken_paragraphs": paragraph_report,
        "broken_formulas": atomic_report["formulas"],
        "broken_tables": atomic_report["tables"],
        "hit_rate_at_3": hit_report_at_3,
        "hit_rate": hit_report,
    }

    print("\n" + "=" * 22 + " Оценка GraphRAG text_units " + "=" * 22)
    print(f"Папка GraphRAG: {report['graph_dir']}")
    print(f"Parquet engine: {report['parquet_engine']}")
    print(f"Количество чанков: {report['chunks_total']}")
    print(f"Средний размер чанка: {report['mean_size_tokens']:.1f} токенов")
    print(
        f"Стандартное отклонение размера: {report['std_size_tokens']:.1f} токенов | "
        f"limit <= {GRAPHRAG_STD_TOKENS_LIMIT} | "
        f"{status_max(report['std_size_tokens'], GRAPHRAG_STD_TOKENS_LIMIT)}"
    )
    print(f"Диапазон размера [min; max]: [{report['min_size_tokens']}; {report['max_size_tokens']}]")
    print(
        f"Доля разорванных предложений: {sentence_report['broken']} / {sentence_report['total']} "
        f"({sentence_report['pct']:.2f}%) | limit <= {GRAPHRAG_BROKEN_SENTENCES_LIMIT}% | "
        f"{status_max(sentence_report['pct'], GRAPHRAG_BROKEN_SENTENCES_LIMIT)}"
    )
    print(
        f"Доля разорванных абзацев: {paragraph_report['broken']} / {paragraph_report['total']} "
        f"({paragraph_report['pct']:.2f}%) | limit <= {GRAPHRAG_BROKEN_PARAGRAPHS_LIMIT}% | "
        f"{status_max(paragraph_report['pct'], GRAPHRAG_BROKEN_PARAGRAPHS_LIMIT)}"
    )
    print(
        f"Разорванные формулы: {atomic_report['formulas']['broken']} / {atomic_report['formulas']['total']} "
        f"({atomic_report['formulas']['pct']:.2f}%) | target {GRAPHRAG_BROKEN_ATOMIC_TARGET:.0f}% | "
        f"{status_max(atomic_report['formulas']['pct'], GRAPHRAG_BROKEN_ATOMIC_TARGET)}"
    )
    print(
        f"Разорванные таблицы: {atomic_report['tables']['broken']} / {atomic_report['tables']['total']} "
        f"({atomic_report['tables']['pct']:.2f}%) | target {GRAPHRAG_BROKEN_ATOMIC_TARGET:.0f}% | "
        f"{status_max(atomic_report['tables']['pct'], GRAPHRAG_BROKEN_ATOMIC_TARGET)}"
    )
    print(
        f"Hit Rate@3: {hit_report_at_3['hits']} / {hit_report_at_3['total']} "
        f"({hit_report_at_3['pct']:.2f}%)"
    )
    print(
        f"Hit Rate@{GRAPHRAG_HIT_RATE_TOP_K}: {hit_report['hits']} / {hit_report['total']} "
        f"({hit_report['pct']:.2f}%) | target >= {GRAPHRAG_HIT_RATE_LIMIT:.0f}% | "
        f"{status_min(hit_report['pct'], GRAPHRAG_HIT_RATE_LIMIT)}"
    )

    if sentence_report["examples"]:
        print("Примеры разорванных предложений:", sentence_report["examples"][:3])
    if paragraph_report["examples"]:
        print("Примеры разорванных абзацев:", paragraph_report["examples"][:3])
    if hit_report["misses"]:
        print("Промахи Hit Rate:", hit_report["misses"][:3])
    print("=" * 72)
    return report


graphrag_chunking_report = evaluate_graphrag_text_units()


### Очистка
- Непечатные и управляющие символы (ASCII < 32, кроме \n, \r, \t)
- Лишние пробелы и табуляции
- HTML/XML-теги
- Стоп-слова (с осторожностью для технических текстов)
- Персональные данные (GDPR, ФЗ-152)
- Удаление картинок

In [6]:
import re

CTRL_RE      = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
INVISIBLE_RE = re.compile(r"[\u200b-\u200f\u2028-\u202f\u2060\ufeff\u00ad]")
SPACEY_RE    = re.compile(r"[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]")
MULTISPACE_RE = re.compile(r"[ ]{2,}")
SPACES_NL_RE = re.compile(r"[ \t]+\n|\n[ \t]+")
SPACES_NL_CLEAN_RE = re.compile(r"[ \t]*\n[ \t]*")
MULTINL_RE    = re.compile(r"\n{3,}")
CR_RE         = re.compile(r"\r")
IMG_MASK = re.compile(r"\[\[IMAGE_\d{4}\]\]")

PII_PATTERNS = [
    re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),                              # email
    re.compile(r"\b(?:\+7|8)[\s\-\(]?\d{3}[\s\-\)]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}\b"),  # ru phone
]


def mask_pii(text):
    for pat in PII_PATTERNS:
        text = pat.sub("[PII]", text)
    return text

def count_issues(text):
    ctrl_hits       = CTRL_RE.findall(text)
    invisible_hits  = INVISIBLE_RE.findall(text)
    nbsp_hits       = SPACEY_RE.findall(text)
    multispace_hits = MULTISPACE_RE.findall(text)
    spaces_nl_hits  = SPACES_NL_RE.findall(text)
    multinl_hits    = MULTINL_RE.findall(text)
    pii_pattern1 = PII_PATTERNS[0].findall(text)
    pii_pattern2 = PII_PATTERNS[1].findall(text)

    return {
        "Управляющие символы ASCII": len(ctrl_hits),
        "Невидимые юникод-символы": len(invisible_hits),
        "Юникод-пробелы": len(nbsp_hits),
        "2+ подряд идущих пробелов": len(multispace_hits),
        "Пробелы перед/после перевода строки (висячие пробелы)": len(spaces_nl_hits),
        "3+ подряд идущих перевода строки": len(multinl_hits),
        "ФЗ-152": len(pii_pattern1) + len(pii_pattern2),
    }


def clean_chunk(text):
    text = IMG_MASK.sub("", text)
    text = CTRL_RE.sub("", text)
    text = INVISIBLE_RE.sub("", text)
    text = SPACEY_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text)
    text = SPACES_NL_CLEAN_RE.sub("\n", text)
    text = MULTINL_RE.sub("\n\n", text)
    for pat in PII_PATTERNS:
        text = pat.sub(" ", text)
    return text.strip()

def clean(chunks):
    return [clean_chunk(c["masked_text"]) for c in chunks]

In [7]:
for k, v in count_issues(protected).items():
    print(f"  {k:12}: {v}")
print()

  Управляющие символы ASCII: 0
  Невидимые юникод-символы: 0
  Юникод-пробелы: 0
  2+ подряд идущих пробелов: 14
  Пробелы перед/после перевода строки (висячие пробелы): 447
  3+ подряд идущих перевода строки: 51
  ФЗ-152      : 0



In [8]:
clean_protected = clean_chunk(protected) # has ALL masks

In [9]:
for k, v in count_issues(clean_protected).items():
    print(f"  {k:12}: {v}")
print()

  Управляющие символы ASCII: 0
  Невидимые юникод-символы: 0
  Юникод-пробелы: 0
  2+ подряд идущих пробелов: 0
  Пробелы перед/после перевода строки (висячие пробелы): 0
  3+ подряд идущих перевода строки: 0
  ФЗ-152      : 0



In [10]:
from bert_score import score

P, R, F1 = score(
    cands=[protected],
    refs=[clean_protected],
    lang="ru",
    verbose=False,
)
print(f"BERTScore F1 = {F1.item():.4f}")
print(f"BERTScore Precision = {P.item():.4f}")
print(f"BERTScore Recall = {R.item():.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore F1 = 1.0000
BERTScore Precision = 1.0000
BERTScore Recall = 1.0000


In [ ]:
# Часть 2. OCR-шум: диагностика и метрики на исходном тексте GraphRAG
OCR_GRAPH_DIRS = {
    "Без предобработки": "graphrag_output_combined",
    "С предобработкой": "graphrag_preprocessed_rubert",
}
OCR_EXAMPLE_LIMIT = 8


def resolve_graphrag_output_dir(path):
    candidates = [
        Path(path),
        Path("..") / path,
        Path("..") / ".." / path,
    ]
    for candidate in candidates:
        if (candidate / "input").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Не найдена папка GraphRAG input: {path}")


def read_graphrag_input_text(graph_dir):
    input_dir = graph_dir / "input"
    input_files = sorted(input_dir.glob("*.md")) + sorted(input_dir.glob("*.txt"))
    if not input_files:
        raise FileNotFoundError(f"В папке {input_dir} нет .md/.txt файлов")

    texts = []
    for path in input_files:
        texts.append(path.read_text(encoding="utf-8", errors="ignore"))
    return "\n\n".join(texts), input_files


OCR_NOISE_CHECKS = [
    (
        "OCR-шум/битая кодировка",
        re.compile(r"\ufffd|вЂ|В«|В»|Â|Ð|Ñ"),
    ),
    (
        "Искаженные числовые параметры: десятичная запятая",
        re.compile(r"\b\d+,\d+\b"),
    ),
    (
        "Искаженные числовые параметры: пробелы внутри числа",
        re.compile(r"(?<!\d)\d(?:\s+\d){2,}(?!\d)"),
    ),
    (
        "Служебные метки: номер страницы отдельной строкой",
        re.compile(r"(?im)^\s*(?:page|стр\.?|страница)?\s*\d{1,4}\s*$"),
    ),
    (
        "Служебные метки: номер страницы в тире",
        re.compile(r"(?im)^\s*[-–—]\s*\d{1,4}\s*[-–—]\s*$"),
    ),
    (
        "Единицы измерения: метрическая система",
        re.compile(r"\b\d+(?:[.,]\d+)?\s*(?:мм|см|мкм|км|кг|мг|кН|МПа|ГПа|°C|Дж|кДж|%|[мН](?!\.[А-ЯЁA-Z]))\b|\b(?:МПа|ГПа|кН|°C|Дж|кДж)\b", re.I),
    ),
    (
        "Единицы измерения: англо-американская система",
        re.compile(r"\b\d+(?:[.,]\d+)?\s*(?:inch|in\.|ft|psi|ksi|lb|lbs|°F)\b", re.I),
    ),
]


def collect_regex_examples(text, pattern, limit=OCR_EXAMPLE_LIMIT, context=90):
    examples = []
    count = 0
    for match in pattern.finditer(text):
        count += 1
        if len(examples) >= limit:
            continue
        start = max(0, match.start() - context)
        end = min(len(text), match.end() + context)
        snippet = re.sub(r"\s+", " ", text[start:end]).strip()
        examples.append(snippet)
    return count, examples


def single_dollar_count(value):
    return sum(
        1
        for index, char in enumerate(value)
        if char == "$"
        and (index == 0 or value[index - 1] != "$")
        and (index + 1 == len(value) or value[index + 1] != "$")
    )


def find_unpaired_dollar_examples(text, limit=OCR_EXAMPLE_LIMIT):
    examples = []
    count = 0
    for line_number, line in enumerate(text.splitlines(), start=1):
        if single_dollar_count(line) % 2 == 0:
            continue
        count += 1
        if len(examples) < limit:
            snippet = re.sub(r"\s+", " ", line).strip()
            examples.append(f"строка {line_number}: {snippet[:260]}")
    return count, examples


def find_formula_ocr_digit_examples(text, limit=OCR_EXAMPLE_LIMIT, context=90):
    examples = []
    count = 0
    formula_re = re.compile(r"(?<!\$)\$([^$\n]+?)\$(?!\$)")
    for match in formula_re.finditer(text):
        formula = match.group(1)
        if not re.search(r"\d\s+\d", formula):
            continue
        count += 1
        if len(examples) >= limit:
            continue
        start = max(0, match.start() - context)
        end = min(len(text), match.end() + context)
        snippet = re.sub(r"\s+", " ", text[start:end]).strip()
        examples.append(snippet)
    return count, examples


def find_broken_table_examples(text, limit=OCR_EXAMPLE_LIMIT):
    examples = []
    markdown_blocks = list(re.finditer(r"(?:^\|[^\n]*\|\s*$\n?){2,}", text, re.M))
    for block in markdown_blocks:
        lines = [line for line in block.group(0).splitlines() if line.strip()]
        column_counts = [line.count("|") for line in lines]
        if len(set(column_counts)) > 1:
            snippet = re.sub(r"\s+", " ", block.group(0)).strip()
            examples.append(f"Разное число колонок {column_counts}: {snippet[:240]}")
        if len(examples) >= limit:
            break

    html_open = len(re.findall(r"<table\b", text, re.I))
    html_close = len(re.findall(r"</table>", text, re.I))
    if html_open != html_close:
        examples.append(f"HTML table tags mismatch: <table>={html_open}, </table>={html_close}")

    return len(examples), examples[:limit]


def inspect_ocr_noise(label, text):
    rows = []
    for issue_name, pattern in OCR_NOISE_CHECKS:
        count, examples = collect_regex_examples(text, pattern)
        rows.append({
            "Источник": label,
            "Проблема": issue_name,
            "Количество": count,
            "Примеры": examples,
        })

    formula_checks = [
        ("Формулы: непарные маркеры $", find_unpaired_dollar_examples),
        ("Формулы: OCR-разреженные цифры внутри формулы", find_formula_ocr_digit_examples),
    ]
    for issue_name, checker in formula_checks:
        count, examples = checker(text)
        rows.append({
            "Источник": label,
            "Проблема": issue_name,
            "Количество": count,
            "Примеры": examples,
        })

    table_count, table_examples = find_broken_table_examples(text)
    rows.append({
        "Источник": label,
        "Проблема": "Нарушенные структуры таблиц",
        "Количество": table_count,
        "Примеры": table_examples,
    })
    return rows


def join_spaced_digits(match):
    return re.sub(r"\s+", "", match.group(0))


def clean_ocr_noise_regex(text):
    cleaned = str(text)
    cleaned = cleaned.replace("\u00a0", " ")
    cleaned = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", cleaned)
    cleaned = re.sub(r"(?m)^\s*(?:page|стр\.?|страница)?\s*\d{1,4}\s*$\n?", "", cleaned)
    cleaned = re.sub(r"(?m)^\s*[-–—]\s*\d{1,4}\s*[-–—]\s*$\n?", "", cleaned)
    cleaned = re.sub(r"([A-Za-zА-Яа-яЁё])-\s*\n\s*([A-Za-zА-Яа-яЁё])", r"\1\2", cleaned)
    cleaned = re.sub(r"(?<=\d),(?=\d)", ".", cleaned)
    cleaned = re.sub(r"(?<!\d)\d(?:\s+\d){2,}(?!\d)", join_spaced_digits, cleaned)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


def evaluate_text_preservation(before_text, after_text):
    before_text = str(before_text)
    after_text = str(after_text)

    cer = float(jiwer.cer(before_text, after_text)) * 100 if before_text else 0.0
    wer = float(jiwer.wer(before_text, after_text)) * 100 if before_text else 0.0

    before_tokens = re.findall(r"\b\w+\b", before_text.lower())
    after_tokens = re.findall(r"\b\w+\b", after_text.lower())
    before_counter = Counter(before_tokens)
    after_counter = Counter(after_tokens)
    common = sum((before_counter & after_counter).values())

    precision = common / len(after_tokens) * 100 if after_tokens else 0.0
    recall = common / len(before_tokens) * 100 if before_tokens else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "CER (%)": round(cer, 2),
        "WER (%)": round(wer, 2),
        "Precision (%)": round(precision, 2),
        "Recall (%)": round(recall, 2),
        "F1 (%)": round(f1, 2),
        "Символов до": len(before_text),
        "Символов после": len(after_text),
        "Токенов до": len(before_tokens),
        "Токенов после": len(after_tokens),
    }


ocr_noise_rows = []
ocr_metrics_rows = []
ocr_sources = {}

for ocr_label, ocr_graph_path in OCR_GRAPH_DIRS.items():
    ocr_graph_dir = resolve_graphrag_output_dir(ocr_graph_path)
    ocr_source_text, ocr_input_files = read_graphrag_input_text(ocr_graph_dir)
    ocr_source_label = ocr_label
    ocr_sources[ocr_label] = {
        "graph_dir": ocr_graph_dir,
        "input_files": ocr_input_files,
        "text": ocr_source_text,
    }

    print(f"\n{'=' * 24} {ocr_label} {'=' * 24}")
    print(f"Папка GraphRAG: {ocr_graph_dir}")
    print("Исходные файлы:")
    for path in ocr_input_files:
        print(f"  - {path}")

    ocr_noise_rows.extend(inspect_ocr_noise(ocr_source_label, ocr_source_text))

    ocr_cleaned_source_text = clean_ocr_noise_regex(ocr_source_text)
    ocr_metrics_rows.append({
        "Источник": ocr_source_label,
        **evaluate_text_preservation(ocr_source_text, ocr_cleaned_source_text),
    })

noise_df = pd.DataFrame([
    {key: value for key, value in row.items() if key != "Примеры"}
    for row in ocr_noise_rows
])

print("\nНаличие OCR-проблем в исходных текстах:")
try:
    display(noise_df)
except NameError:
    print(noise_df.to_string(index=False))

print("\nПримеры найденных проблем:")
for row in ocr_noise_rows:
    if not row["Примеры"]:
        continue
    print(f"\n[{row['Источник']}] {row['Проблема']} | найдено: {row['Количество']}")
    for example in row["Примеры"][:OCR_EXAMPLE_LIMIT]:
        print(f"  - {example}")

def show_dataframe(df):
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def build_ocr_problem_comparison(noise_rows, before_label="Без предобработки", after_label="С предобработкой"):
    problems = sorted({row["Проблема"] for row in noise_rows})
    rows = []
    for problem in problems:
        before_count = next(
            (row["Количество"] for row in noise_rows if row["Источник"] == before_label and row["Проблема"] == problem),
            0,
        )
        after_count = next(
            (row["Количество"] for row in noise_rows if row["Источник"] == after_label and row["Проблема"] == problem),
            0,
        )
        delta = after_count - before_count
        reduction = (before_count - after_count) / before_count * 100 if before_count else None
        rows.append({
            "Проблема": problem,
            "До предобработки": before_count,
            "После предобработки": after_count,
            "Изменение": delta,
            "Снижение (%)": round(reduction, 2) if reduction is not None else None,
        })
    return pd.DataFrame(rows)


regex_metrics_df = pd.DataFrame(ocr_metrics_rows)

print("\nМетрики дополнительной regex-очистки внутри каждого текста")
print("Важно: это НЕ прямое сравнение до/после предобработки. Нули в строке после предобработки означают, что дополнительная regex-очистка уже ничего не изменила.")
show_dataframe(regex_metrics_df)

before_label = "Без предобработки"
after_label = "С предобработкой"
before_text = ocr_sources[before_label]["text"]
after_text = ocr_sources[after_label]["text"]

preprocessing_metrics_df = pd.DataFrame([{
    "Сравнение": f"{before_label} -> {after_label}",
    **evaluate_text_preservation(before_text, after_text),
}])

print("\nПрямое сравнение текста ДО предобработки и ПОСЛЕ предобработки")
print("CER/WER здесь показывают величину изменения текста; Precision/Recall/F1 показывают сохранность слов относительно исходного текста.")
show_dataframe(preprocessing_metrics_df)

ocr_problem_comparison_df = build_ocr_problem_comparison(ocr_noise_rows, before_label, after_label)

print("\nСравнение найденных OCR-проблем ДО и ПОСЛЕ предобработки")
show_dataframe(ocr_problem_comparison_df)

### Нормализация

Рекомендация: нормализация дат, единиц измерения, формул, таблиц в одном стиле

Нормализация единиц измерения: приведение к СИ через Pint.
Нормализация дат: → ISO 8601 (2025-03-12) через dateutil.
Нормализация чисел: единый формат, научная нотация → десятичная.
Нормализация формул: LaTeX → канонический вид через SymPy, сохранение AST.
Нормализация таблиц: единая структура, типы данных через pandas.

In [ ]:
pip install markdown

In [11]:
import markdown


def markdown_table_to_html(text: str) -> str:
    return markdown.markdown(
        text,
        extensions=["tables"]
    )
def convert_markdown_tables_in_store(store, tables_dict, debug=False):
    ok = skipped = failed = 0
    for key in tables_dict:
        value = store.get(key)
        if not isinstance(value, str):
            skipped += 1
            continue

        v = value.lstrip()

        if v.startswith("<"):
            skipped += 1
            continue
        if not v.startswith("|"):
            skipped += 1
            continue

        try:
            html = markdown_table_to_html(value)
            if html and html.strip():
                store[key] = html.strip()
                ok += 1
            else:
                failed += 1
                if debug:
                    print(f"{key}: markdown вернул пусто")
        except Exception as e:
            failed += 1
            if debug:
                print(f"{key}: {type(e).__name__}: {e}")

In [12]:
convert_markdown_tables_in_store(STORE, TABLES, debug=True)

In [13]:
STORE

{'[[FORMULA_0000]]': '$$\nE = 2 \\lim_{\\Delta l \\to 0} \\sum_{0}^{l} A_{\\mathrm{yx}} \\cdot V_{\\mathrm{zx},\\mathrm{zx}} = \\frac{2}{t} \\cdot \\int_{0}^{l} A_{\\mathrm{yx}}(l) \\cdot S(l)   dl \\cdot \\left[ \\frac{\\Delta l}{\\mathrm{CM}^2} \\right].\n$$',
 '[[FORMULA_0001]]': '$\\mathrm{OT}+20\\mathrm{~\\pi}0-196^{\\mathrm{~0}}\\mathrm{C}$',
 '[[FORMULA_0002]]': '$4 5 0 \\times 3 3 0 \\times 2 7 { , } 7$',
 '[[FORMULA_0003]]': '$5 5 0 \\times 3 3 0 \\times 2 3$',
 '[[FORMULA_0004]]': '$1 9 6 ~ ^ { \\circ } \\mathrm { C } _ { 3 }$',
 '[[FORMULA_0005]]': '$^ { 0 , 2 }$',
 '[[FORMULA_0006]]': '$( \\mathrm{M} \\bot \\mathrm{sw/M}^2 )$',
 '[[FORMULA_0007]]': '$( \\Gamma _ { \\mathrm { {   H C I } } } )$',
 '[[FORMULA_0008]]': '$( \\times ~ 5 ~ \\textbar { - } ~ 1 0 )$',
 '[[FORMULA_0009]]': '$8 0 ~ ^ { \\circ } \\mathrm { C }$',
 '[[FORMULA_0010]]': '$\\left( \\Delta = \\mathrm{KCU_{max}} - \\mathrm{KCU_{min}} \\right)$',
 '[[FORMULA_0011]]': '$175\\  五  \\times /  cm ^2$',
 '[[FORMU

In [ ]:
!pip install sympy

In [ ]:
!pip install "antlr4-python3-runtime==4.11"

In [14]:
import re

def preprocess_formula(store,formulas_store):
    def normalize(text):
        text = text.strip()
        text = re.sub(r"^\${1,2}|\${1,2}$", "", text)
        text = re.sub(r"[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]", " ", text)
        text = re.sub(r"\\(?:,|;|:|!| )", "", text)
        text = text.replace("~", "")
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s*([{}_^])\s*", r"\1", text)
        text = re.sub(r"(?<=\d)\s+(?=\d)", "", text)
        text = re.sub(r"\\([a-zA-Z]+)\s*\{", r"\\\1{", text)
        text = re.sub(r"\{\s*([^{}]*?)\s*\}", r"{\1}", text)
        return text.strip()

    for key in formulas_store:
        store[key] = normalize(formulas_store[key])


In [15]:
preprocess_formula(STORE, FORMULAS)

In [16]:
from dateutil import parser


def normalize_dates(store, dates_store) -> tuple[str, int]:
    def replace_date(value):
        try:
            if re.match(r"^\d{1,2}[./-]\d{1,2}[./-]\d{4}$", value):
                date = parser.parse(value, dayfirst=True)
            else:
                date = parser.parse(value)
            return date.strftime("%Y-%m-%d")
        except (ValueError, OverflowError):
            return value

    for key in dates_store:
        store[key] = replace_date(dates_store[key])

In [17]:
normalize_dates(STORE, DATES)

In [18]:
from decimal import Decimal


def normalize_numbers(store, numbers_store) -> tuple[str, int]:
    def replace_number(value):
        value = value.replace(" ", "").replace("_", "")
        value = value.replace(",", ".")

        try:
            number = Decimal(value)
            result = format(number, "f")
            if "." in result:
                result = result.rstrip("0").rstrip(".")

            if result in ("-0", "+0"):
                result = "0"
            return result
        except Exception:
            pass

    for key in numbers_store:
        store[key] = replace_number(numbers_store[key])

In [19]:
normalize_numbers(STORE, NUMBERS)

In [20]:
STORE

{'[[FORMULA_0000]]': 'E = 2 \\lim_{\\Delta l \\to 0}\\sum_{0}^{l}A_{\\mathrm{yx}}\\cdot V_{\\mathrm{zx},\\mathrm{zx}}= \\frac{2}{t}\\cdot \\int_{0}^{l}A_{\\mathrm{yx}}(l) \\cdot S(l) dl \\cdot \\left[ \\frac{\\Delta l}{\\mathrm{CM}^2}\\right].',
 '[[FORMULA_0001]]': '\\mathrm{OT}+20\\mathrm{\\pi}0-196^{\\mathrm{0}}\\mathrm{C}',
 '[[FORMULA_0002]]': '450 \\times 330 \\times 27{,}7',
 '[[FORMULA_0003]]': '550 \\times 330 \\times 23',
 '[[FORMULA_0004]]': '196^{\\circ}\\mathrm{C}_{3}',
 '[[FORMULA_0005]]': '^{0 , 2}',
 '[[FORMULA_0006]]': '( \\mathrm{M}\\bot \\mathrm{sw/M}^2 )',
 '[[FORMULA_0007]]': '( \\Gamma_{\\mathrm{{H C I}}})',
 '[[FORMULA_0008]]': '( \\times 5 \\textbar{-}10 )',
 '[[FORMULA_0009]]': '80^{\\circ}\\mathrm{C}',
 '[[FORMULA_0010]]': '\\left( \\Delta = \\mathrm{KCU_{max}}- \\mathrm{KCU_{min}}\\right)',
 '[[FORMULA_0011]]': '175 五 \\times / cm^2',
 '[[FORMULA_0012]]': '\\pi 0 - 60 \\mathrm{C})',
 '[[FORMULA_0013]]': 'r_{0}',
 '[[FORMULA_0014]]': 'r_{0}',
 '[[FORMULA_0015]

In [21]:
Path("clean_protected.md").write_text(clean_protected, encoding="utf-8")

85598

In [28]:
import pandas as pd
from pathlib import Path

def inspect_parquet(path: str, n_examples: int = 2):
    """Читает parquet, печатает инфо о колонках и примеры строк."""
    path = Path(path)
    if not path.exists():
        print(f"Файл не найден: {path}")
        return None

    df = pd.read_parquet(path, engine="fastparquet")

    print("=" * 70)
    print(f"Файл: {path.name}")
    print(f"Размер: {path.stat().st_size / 1024:.1f} KB")
    print(f"Строк: {len(df)}, Колонок: {len(df.columns)}")
    print("=" * 70)

    # ─── колонки ───
    print("\nКОЛОНКИ:")
    print(f"{'#':>3}  {'имя':<30} {'dtype':<20} {'non-null':>10}  пример")
    print("-" * 100)

    for i, col in enumerate(df.columns):
        dtype = str(df[col].dtype)
        non_null = df[col].notna().sum()

        # первый непустой пример
        sample = df[col].dropna()
        if len(sample):
            val = sample.iloc[0]
            # обрезаем длинные значения
            if isinstance(val, (list, tuple)):
                sample_str = f"[{type(val).__name__} len={len(val)}]"
            elif isinstance(val, str):
                sample_str = val[:40].replace("\n", " ")
                if len(val) > 40:
                    sample_str += "..."
            else:
                sample_str = str(val)[:40]
        else:
            sample_str = "(пусто)"

        print(f"{i:>3}  {col:<30} {dtype:<20} {non_null:>10}  {sample_str}")

    # ─── примеры строк ───
    print(f"\nПРИМЕРЫ СТРОК (первые {n_examples}):")
    print("=" * 70)

    for idx in range(min(n_examples, len(df))):
        row = df.iloc[idx]
        print(f"\n--- Строка #{idx} ---")
        for col in df.columns:
            val = row[col]

            # форматирование для читаемости
            if isinstance(val, (list, tuple)):
                # для списков — показываем длину и первые элементы
                n = len(val)
                if n and isinstance(val[0], (int, float)):
                    display = f"[{n} элементов] {val[:5]}..."
                elif n and isinstance(val[0], (list, tuple)):
                    display = f"[{n} векторов] первый: {val[0][:5]}..."
                else:
                    display = f"[{n} элементов] {val[:3]}"
            elif isinstance(val, str):
                if len(val) > 200:
                    display = val[:200] + "..."
                else:
                    display = val
            else:
                display = val

            print(f"  {col}: {display}")

    print()
    return df


DEFAULT = "text_units.parquet"

path = DEFAULT
inspect_parquet(path, n_examples=2)

Файл: text_units.parquet
Размер: 77.9 KB
Строк: 28, Колонок: 4

КОЛОНКИ:
  #  имя                            dtype                  non-null  пример
----------------------------------------------------------------------------------------------------
  0  id                             object                       28  dbe56ea9c0604a0c7d395b56a41ea9176e3c523a...
  1  document_id                    object                       28  7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed...
  2  text                           object                       28  На правах рукописи  Скородумов Сергей Ва...
  3  n_tokens                       int64                        28  1024

ПРИМЕРЫ СТРОК (первые 2):

--- Строка #0 ---
  id: dbe56ea9c0604a0c7d395b56a41ea9176e3c523a919140a40810a9a59358f889a92bc362f0eba36de2b02221c50502102edc1bfa5002d8a821fca53bb3545eda
  document_id: 7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02882e328c72b4d1cf4ace724bb8d3b206a6e9dfe258b04fc022a8897520a5e2ee7cb3cec6a8ad98e975
  text: На п

,id,document_id,text,n_tokens
0,dbe56ea9c0604a0c7d395b56a41ea9176e3c523a919140...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,На правах рукописи\n\nСкородумов Сергей Валери...,1024
1,07dc6b97e6925a74e7ae33d26f23e808c45c515f037875...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,температур испытаний [[FORMULA_0001]]\n\nМасс...,1024
2,f767950d881ac103547cf2759b4a91b3820a630502b892...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,для сериальных испытаний и по\n\n[[NUMBER_002...,1024
3,9298e578310b064ea20ba4cfef3391c1fa43bec4eea19f...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,термической обработки и ударная вязкость осно...,1024
4,c523d2d2823b34876abe57f916a71f065e5d5bf32b502e...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,развитой структурной неоднородности. Например...,1024
5,7a2e9e8563c6a92c9f80ea97d7bc70749e9e7eeedefbef...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,) сопоставление диаграммы растяжения с сигнала...,1024
6,d243015ab293381369df619bfc1837a4987256d92bd3b9...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,"ов. Например, для стали 40Х2Н2МА в улучшенном ...",1024
7,b31f4330e29bb507b95a9c49fda5d4bb80e543c4c0cacc...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,многообразие механизмов разрушения определяет...,1024
8,e93deda3a32778e378e55d4ffc55ecd872209501e74f8d...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,необходимость анализа их «текстуры». С этой ц...,1024
9,25939735262ec9bd3361c07f1c2ebc72db9d036eacb3d4...,7c3cc6ee49bbd468d5bf0149a4d204d704ebc4ed28cc02...,"урная полосчатость,\n\n[[NUMBER_0152]]\n\n) ми...",1024


In [29]:
for key in list(STORE.keys()):
    if key.startswith("[[FORMULA_"):
        value = STORE[key]
        if isinstance(value, str):
            v = value.strip("$ \n")
            STORE[key] = f"$${v}$$"

In [31]:
import pandas as pd
from pathlib import Path

TEXT_UNITS = "text_units.parquet"


def replace_markers(text: str, store: dict) -> str:
    """Заменяет все маркеры [[TYPE_NNNN]] на значения из STORE."""
    for key, value in store.items():
        if key in text:
            text = text.replace(key, value)
    return text


def restore_text_units(path: str, store: dict, keep_masked: bool = True):
    """
    Читает text_units.parquet, заменяет маркеры на значения из STORE.
    Если keep_masked=True — сохраняет исходный текст в колонку text_masked.
    """
    df = pd.read_parquet(path, engine="fastparquet")

    print(f"Чанков до замены: {len(df)}")
    print(f"Колонки: {list(df.columns)}")

    # сохраняем masked-версию
    if keep_masked:
        df["text_masked"] = df["text"]

    # считаем сколько маркеров было
    def count_markers(t):
        import re
        return len(re.findall(r"\[\[(?:FORMULA|TABLE|DATE|NUMBER)_\d{4}\]\]", t))

    before = df["text"].apply(count_markers).sum()

    # заменяем
    df["text"] = df["text"].apply(lambda t: replace_markers(t, store))

    # считаем сколько осталось
    after = df["text"].apply(count_markers).sum()

    print(f"Маркеров до замены: {before}")
    print(f"Маркеров после:     {after}")

    # сохраняем
    #df.to_parquet(path)

    print(f"Сохранено: {path}")
    print()
    print("Пример первого чанка до:")
    print(df.iloc[5]["text_masked"][:200] if keep_masked else "—")
    print()
    print("Пример первого чанка после:")
    print(df.iloc[5]["text"][:200])

    return df


# ─── использование ───
# STORE — ваш словарь {[[FORMULA_0000]]: "...", [[TABLE_0000]]: "..."}
df = restore_text_units(TEXT_UNITS, STORE, keep_masked=True)

Чанков до замены: 28
Колонки: ['id', 'document_id', 'text', 'n_tokens']
Маркеров до замены: 500
Маркеров после:     0
Сохранено: text_units.parquet

Пример первого чанка до:
) сопоставление диаграммы растяжения с сигналами АЭ

Такие испытания на растяжение (в сочетании с фрактографией) показали, что после сужения в шейку разрушение идет либо по «слабым» поверхностям с обр

Пример первого чанка после:
) сопоставление диаграммы растяжения с сигналами АЭ

Такие испытания на растяжение (в сочетании с фрактографией) показали, что после сужения в шейку разрушение идет либо по «слабым» поверхностям с обр


In [32]:
df.to_parquet("test.parquet")

In [33]:
# Final: restore tables and formulas
clean_restored_chunks = restore(clean_chunk)

TypeError: argument of type 'function' is not iterable

### Токенизация (Оценка)

Рекомендлация: по предложениям дальше по словам

In [38]:
import re
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer


TEXT_UNITS = "text_units.parquet"
MODEL_NAME = "cointegrated/rubert-tiny2"

MARKER_RE = re.compile(r"\[\[(FORMULA|TABLE|DATE|NUMBER|IMAGE)_(\d{4})\]\]")
FORMULA_BLOCK_RE = re.compile(r"\$\$.*?\$\$", re.DOTALL)
FORMULA_INLINE_RE = re.compile(r"(?<!\$)\$([^$\n]+?)\$(?!\$)")
WORD_RE = re.compile(r"[А-Яа-яЁёA-Za-z]+(?:[-/][А-Яа-яЁёA-Za-z]+)*", re.UNICODE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def restore_text(text, store):
    def repl(m):
        kind, num = m.group(1), m.group(2)
        key = f"[[{kind}_{num}]]"
        value = store.get(key)
        if value is None:
            return m.group(0)
        if kind == "FORMULA":
            v = value if isinstance(value, str) else value.get("normalized", "")
            v = v.strip("$ \n")
            return f"$${v}$$"
        if kind == "TABLE":
            return value if isinstance(value, str) else value.get("html", "")
        return value if isinstance(value, str) else str(value)
    return MARKER_RE.sub(repl, text)


def formula_metrics(texts):
    tokens = []
    examples = []
    for text in texts:
        for m in FORMULA_BLOCK_RE.finditer(text):
            pieces = tokenizer.tokenize(m.group(0))
            tokens.append(len(pieces))
            if len(examples) < 5:
                examples.append((m.group(0)[:60], len(pieces)))
    return {
        "total": len(tokens),
        "avg":   sum(tokens) / len(tokens) if tokens else 0,
        "max":   max(tokens) if tokens else 0,
        "examples": examples,
    }


def oov_rate(texts, threshold=3):
    total = oov = 0
    oov_words = Counter()

    for text in texts:
        clean = MARKER_RE.sub(" ", text)
        clean = FORMULA_BLOCK_RE.sub(" ", clean)
        clean = FORMULA_INLINE_RE.sub(" ", clean)
        for w in WORD_RE.findall(clean):
            total += 1
            if len(tokenizer.tokenize(w)) >= threshold:
                oov += 1
                oov_words[w] += 1

    return {
        "total": total,
        "oov": oov,
        "rate": oov / total if total else 0.0,
        "top": oov_words.most_common(15),
    }


df = pd.read_parquet(TEXT_UNITS, engine="fastparquet")
restored = [restore_text(t, STORE) for t in df["text"].tolist()]

print("Пример:")
print(f"  {restored[1][:200]}\n")

print("── ФОРМУЛЫ ──")
fm = formula_metrics(restored)
print(f"  всего формул:       {fm['total']}")
print(f"  avg токенов:        {fm['avg']:.2f}")
print(f"  max токенов:        {fm['max']}")
for txt, n in fm["examples"]:
    print(f"           {txt!r} → {n} токенов")

print()
print("── OOV RATE ──")
ov = oov_rate(restored)
print(f"  слов всего={ov['total']} "
      f"oov={ov['oov']} "
      f"rate={ov['rate']*100:.2f}%")
if ov["top"]:
    print(f"  топ OOV: {ov['top']}")

Пример:
   остаточного ресурса изделий. Однако преимущественно качественный характер ранжировки структур и вязкости затрудняют оценку механизмов разрушения разномасштабных структур, закономерностей их эволюции 

── ФОРМУЛЫ ──
  всего формул:       1
  avg токенов:        133.00
  max токенов:        133
           '$$\nE = 2 \\lim_{\\Delta l \\to 0} \\sum_{0}^{l} A_{\\mathrm{yx}} ' → 133 токенов

── OOV RATE ──
  слов всего=12739 oov=1049 rate=8.23%
  топ OOV: [('микроструктуры', 39), ('неоднородности', 23), ('Скородумов', 22), ('МИСиС', 21), ('изломов', 21), ('НМФА', 20), ('микроструктур', 17), ('Экономная', 16), ('трещиностойкости', 15), ('colspan', 13), ('неметаллических', 12), ('Пышминцев', 12), ('разномасштабных', 11), ('излома', 11), ('rowspan', 11)]


### Векторизация (e5)

In [37]:
import lancedb
import numpy as np

db = lancedb.connect("/Users/alisaegorova/PycharmProjects/Graphs&GraphModels/Avtoreferat_Skorodumov_S.V./lancedb")

for table_name in ["entity_description", "text_unit_text", "community_full_content"]:
    df = db.open_table(table_name).to_pandas()
    vectors = np.vstack(df["vector"].tolist())
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors_norm = vectors / (norms + 1e-12)

    sim = vectors_norm @ vectors_norm.T
    n = len(vectors)
    mask = ~np.eye(n, dtype=bool)

    print(f"── {table_name} ──")
    print(f"  строк: {n}, размерность: {vectors.shape[1]}")
    print(f"  косинус: min={sim[mask].min():.3f} "
          f"avg={sim[mask].mean():.3f} "
          f"max={sim[mask].max():.3f}")
    print()

ModuleNotFoundError: No module named 'lancedb'

In [ ]:
# Final: restore tables and formulas
clean_restored_chunks = [restore(clean_chunk) for clean_chunk in clean_chunks]

## Сравнение графов GraphRAG

Код ниже анализирует уже построенные GraphRAG-графы: без повторного извлечения сущностей и ребер.


In [ ]:
# Сравнение/оценка готовых GraphRAG-графов без повторного построения
GRAPH_DIRS = {
    # Если появится второй граф, добавь его сюда отдельной строкой.
    "Без предобработки": "graphrag_output_combined",
    "С предобработкой": "graphrag_preprocessed_rubert",
}

DOMAIN_TERMS = [
    "К60", "К65", "Х80", "X80", "CTOD", "TMCP", "HTP", "СЭМ", "EBSD",
    "15Х2НМФА", "12Х2Н2МА", "40Х2Н2МА", "38ХН3МФА",
    "08Г2МБ", "07Г2МБ", "06Г2НДБ", "06Г2МФБ", "16Г2АФ",
    "конструкционные стали", "трубные стали", "сварные соединения",
    "термическая обработка", "термомеханическая обработка",
    "вязкое разрушение", "трещиностойкость", "энергоемкость разрушения", "ударная вязкость",
    "микроструктура", "феррито-бейнитная", "бейнит", "аустенит",
    "локальная хрупкость", "участок крупного зерна",
    "Шарпи", "полиэдры Вороного", "Voronoi",
]
# Отдельная проверка сохранности первых терминов: от К60 до 16Г2АФ включительно.
PRIMARY_DOMAIN_TERMS = DOMAIN_TERMS[:18]

RELATION_RULES = {
    "affiliation/location": [
        "located", "располож", "наход", "город", "университет", "candidate at",
        "defense", "защит", "совет",
    ],
    "authorship/publication": [
        "presented", "conference", "конференц", "публикац", "proceedings",
        "journal", "работа", "автор", "диссертац",
    ],
    "material/grade": [
        "steel", "сталь", "труб", "к60", "к65", "x80", "х80", "grade", "class",
    ],
    "process/method": [
        "processed", "method", "использ", "метод", "tmcp", "htp", "испытан",
        "обработ", "измер", "анализ",
    ],
    "property/phenomenon": [
        "property", "resistance", "вязк", "трещ", "разруш", "микрострукт",
        "ctod", "энергоем", "хрупк",
    ],
}


def resolve_graph_dir(path):
    candidates = [
        Path(path),
        Path("..") / path,
        Path("..") / ".." / path,
    ]
    for candidate in candidates:
        if (candidate / "output" / "graph.graphml").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Не найден GraphRAG output: {path}")


def normalize_text(value):
    value = str(value or "").replace("Ё", "Е").replace("ё", "е")
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def normalize_key(value):
    value = normalize_text(value).upper()
    value = re.sub(r"[«»\"'`]", "", value)
    value = re.sub(r"\b(ОАО|ООО|АО|ФГАОУ|НИТУ|ФГБОУ|ИМ\.?|ИМЕНИ)\b", " ", value)
    value = re.sub(r"[^0-9A-ZА-ЯЁ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def compact_key(value):
    return re.sub(r"\s+", "", normalize_key(value))


def read_source_text(graph_dir):
    input_dir = graph_dir / "input"
    texts = []
    for path in sorted(input_dir.glob("*.md")) + sorted(input_dir.glob("*.txt")):
        texts.append(path.read_text(encoding="utf-8", errors="ignore"))
    return "\n\n".join(texts)


def parse_cache_records(graph_dir):
    entity_types = defaultdict(Counter)
    entity_descriptions = defaultdict(list)
    edge_descriptions = defaultdict(list)
    cache_dir = graph_dir / "cache" / "extract_graph"
    if not cache_dir.exists():
        return entity_types, entity_descriptions, edge_descriptions

    def _extract_content(payload):
        if isinstance(payload, dict):
            for key in ("content", "formatted_response"):
                if isinstance(payload.get(key), str):
                    return payload[key]
            response = payload.get("response")
            if isinstance(response, dict):
                if isinstance(response.get("content"), str):
                    return response["content"]
                choices = response.get("choices") or []
                if choices:
                    message = choices[0].get("message", {})
                    if isinstance(message.get("content"), str):
                        return message["content"]
            result = payload.get("result")
            if isinstance(result, dict):
                return _extract_content(result)
        return ""

    entity_re = re.compile(r'\("?entity"?<\|>([^<\n]+?)<\|>([^<\n]+?)<\|>(.*?)\)', re.I)
    rel_re = re.compile(r'\("?relationship"?<\|>([^<\n]+?)<\|>([^<\n]+?)<\|>(.*?)<\|>(\d+(?:\.\d+)?)\)', re.I)

    for path in cache_dir.iterdir():
        if not path.is_file():
            continue
        try:
            content = _extract_content(json.loads(path.read_text(encoding="utf-8", errors="ignore")))
        except Exception:
            content = path.read_text(encoding="utf-8", errors="ignore")

        for match in entity_re.finditer(content):
            title, entity_type, description = [normalize_text(item).strip(" *>\")") for item in match.groups()]
            if title:
                entity_types[normalize_key(title)][entity_type.upper()] += 1
                entity_descriptions[normalize_key(title)].append(description)

        for match in rel_re.finditer(content):
            source, target, description, _weight = [normalize_text(item).strip(" *>\")") for item in match.groups()]
            if source and target:
                key = tuple(sorted((normalize_key(source), normalize_key(target))))
                edge_descriptions[key].append(description)

    return entity_types, entity_descriptions, edge_descriptions


def get_entity_type(node, type_votes):
    votes = type_votes.get(normalize_key(node))
    if votes:
        return votes.most_common(1)[0][0]

    value = normalize_key(node)
    if re.search(r"\b(К60|К65|X80|Х80|СТАЛ|STEEL|TMCP|HTP|CTOD|СЭМ|EBSD)\b", value):
        return "TECHNICAL_TERM"
    if re.search(r"\b(МОСКВА|ЧЕЛЯБИНСК|УХТА|БЕЛЬГИЯ|BRAZIL|ITALY|RUSSIA|РОССИЯ)\b", value):
        return "GEO"
    if re.search(r"\b(УНИВЕРСИТЕТ|МИСИС|РОСНИТИ|ГАЗПРОМ|ЗАВОД|ИНСТИТУТ|СОВЕТ|COMPANY)\b", value):
        return "ORGANIZATION"
    if re.search(r"[А-ЯA-Z][А-ЯA-Z]+ [А-ЯA-Z]\.?[А-ЯA-Z]?\.?$", value):
        return "PERSON"
    return "UNKNOWN"


def classify_relation(source, target, descriptions):
    text = " ".join([source, target] + descriptions).lower()
    for relation_type, markers in RELATION_RULES.items():
        if any(marker in text for marker in markers):
            return relation_type
    return "other"


def node_in_text(node, source_text):
    node_norm = normalize_key(node)
    text_norm = normalize_key(source_text)
    if node_norm and node_norm in text_norm:
        return True
    parts = [part for part in node_norm.split() if len(part) >= 3]
    return len(parts) >= 2 and all(part in text_norm for part in parts[:3])


def edge_supported(source, target, source_text, window=600):
    source_key = compact_key(source)
    target_key = compact_key(target)
    text_key = compact_key(source_text)
    positions_a = [m.start() for m in re.finditer(re.escape(source_key), text_key)]
    positions_b = [m.start() for m in re.finditer(re.escape(target_key), text_key)]
    if not positions_a or not positions_b:
        return False
    return any(abs(a - b) <= window for a in positions_a[:50] for b in positions_b[:50])


def is_noisy(value):
    value = str(value or "")
    stripped = value.strip()
    if not stripped:
        return True
    if re.search(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f�]", stripped):
        return True
    if re.search(r"(base64|data:image|<table|</td>|</tr>|\\[a-zA-Z]+)", stripped, re.I):
        return True
    if len(stripped) <= 2:
        return True
    non_word = len(re.findall(r"[^0-9A-Za-zА-Яа-яЁё\s]", stripped))
    return non_word / max(1, len(stripped)) > 0.35


def detect_coreference_groups(nodes):
    buckets = defaultdict(list)
    for node in nodes:
        key = normalize_key(node)
        if not key:
            continue
        buckets[key].append(node)

        person_match = re.match(r"^([А-ЯA-Z][А-ЯA-Z]+)\s+([А-ЯA-Z])[А-ЯA-Z]*\s+([А-ЯA-Z])[А-ЯA-Z]*$", key)
        if person_match:
            surname, first, patronymic = person_match.groups()
            buckets[f"PERSON:{surname} {first} {patronymic}"].append(node)
        initials_match = re.match(r"^([А-ЯA-Z][А-ЯA-Z]+)\s+([А-ЯA-Z])\s*([А-ЯA-Z])$", key)
        if initials_match:
            surname, first, patronymic = initials_match.groups()
            buckets[f"PERSON:{surname} {first} {patronymic}"].append(node)

        acronym = re.sub(r"[^А-ЯA-Z0-9]", "", key)
        if 3 <= len(acronym) <= 12:
            buckets[f"ACRONYM:{acronym}"].append(node)

    groups = []
    seen = set()
    for values in buckets.values():
        unique = sorted(set(values))
        if len(unique) < 2:
            continue
        marker = tuple(unique)
        if marker not in seen:
            seen.add(marker)
            groups.append(unique)
    return groups


def parse_id_list(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [str(item).strip() for item in value if str(item).strip()]
    value = str(value).strip()
    if not value or value.lower() == "nan":
        return []
    value = value.strip("[]")
    value = value.replace("\n", ",")
    items = [item.strip().strip("'\"") for item in value.split(",")]
    return [item for item in items if item]


def collect_text_unit_ids(df, columns=("text_unit_ids", "source_chunk_ids")):
    ids = set()
    for column in columns:
        if column not in df.columns:
            continue
        for value in df[column]:
            ids.update(parse_id_list(value))
    return ids


def source_integrity(source_text, graph_text):
    formula_re = re.compile(r"\$\$.*?\$\$|(?<!\$)\$[^\$\n]+?\$(?!\$)", re.DOTALL)
    source_formulas = formula_re.findall(source_text)
    source_tables = re.findall(r"<table\b.*?</table>|(?:^\|[^\n]*\|\s*$\n?){2,}", source_text, re.I | re.M | re.DOTALL)
    source_images = re.findall(r"!\[[^\]]*\]\([^)]+\)", source_text)
    graph_formula_fragments = re.findall(r"\$[^$\n]*(?:$|\n)|\\[a-zA-Z]+\s*[{^_]", graph_text)
    graph_table_fragments = re.findall(r"</?(?:table|tr|td|tbody|th)\b|(?:\|\s*){3,}", graph_text, re.I)
    return {
        "source_formulas": len(source_formulas),
        "source_tables": len(source_tables),
        "source_images": len(source_images),
        "graph_formula_fragments": len(graph_formula_fragments),
        "graph_table_fragments": len(graph_table_fragments),
        "integrity_issues": len(graph_formula_fragments) + len(graph_table_fragments),
    }


def build_adjacency(nodes, edges):
    adjacency = {node: set() for node in nodes}
    for source, target, _data in edges:
        adjacency.setdefault(source, set()).add(target)
        adjacency.setdefault(target, set()).add(source)
    return adjacency


def dfs_connected_components(adjacency):
    visited = set()
    components = []

    for start in adjacency:
        if start in visited:
            continue

        component = set()
        stack = [start]
        visited.add(start)

        while stack:
            node = stack.pop()
            component.add(node)
            for neighbor in adjacency[node]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)

        components.append(component)

    return components


def dfs_bridges_tarjan(adjacency):
    timer = 0
    tin = {}
    low = {}
    visited = set()
    bridges = []

    def dfs(node, parent=None):
        nonlocal timer
        visited.add(node)
        tin[node] = low[node] = timer
        timer += 1

        for neighbor in adjacency[node]:
            if neighbor == parent:
                continue
            if neighbor in visited:
                low[node] = min(low[node], tin[neighbor])
                continue

            dfs(neighbor, node)
            low[node] = min(low[node], low[neighbor])
            if low[neighbor] > tin[node]:
                bridges.append(tuple(sorted((node, neighbor))))

    for node in adjacency:
        if node not in visited:
            dfs(node)

    return bridges


def manual_degrees(adjacency):
    degrees = {}
    for node, neighbors in adjacency.items():
        # Петля считается как две инцидентности, как в классической степени вершины.
        degrees[node] = len(neighbors) + int(node in neighbors)
    return degrees


def manual_graph_metrics(nodes, edges):
    adjacency = build_adjacency(nodes, edges)
    components = dfs_connected_components(adjacency)
    bridges = dfs_bridges_tarjan(adjacency)
    degrees = manual_degrees(adjacency)

    node_count = len(adjacency)
    edge_count = len(edges)
    component_count = len(components)
    degrees_of_freedom = edge_count - node_count + component_count

    return {
        "algorithm": "ручной DFS; мосты - Tarjan DFS по tin/low",
        "components": components,
        "largest_component": max((len(component) for component in components), default=0),
        "bridges": bridges,
        "degrees": degrees,
        "degrees_of_freedom": degrees_of_freedom,
        # Для неориентированного графа число независимых циклов равно цикломатическому числу.
        "cycles_count": max(0, degrees_of_freedom),
    }


def analyze_graph(label, graph_dir):
    graph_dir = resolve_graph_dir(graph_dir)
    output_dir = graph_dir / "output"
    entities_df = pd.read_parquet(output_dir / "entities.parquet", engine="fastparquet")
    relationships_df = pd.read_parquet(output_dir / "relationships.parquet", engine="fastparquet")
    text_units_df = pd.read_parquet(output_dir / "text_units.parquet", engine="fastparquet")

    # Для количественных метрик берем все сущности из parquet, включая изолированные.
    # graph.graphml может не содержать вершины без ребер, поэтому он занижает nodes/components.
    node_seen = set()
    nodes = []
    for title in entities_df.get("title", pd.Series(dtype=str)).fillna("").astype(str):
        title = normalize_text(title)
        if title and title not in node_seen:
            node_seen.add(title)
            nodes.append(title)

    edges = []
    orphan_endpoint_nodes = []
    for _idx, row in relationships_df.iterrows():
        source = normalize_text(row.get("source", ""))
        target = normalize_text(row.get("target", ""))
        if not source or not target:
            continue
        edges.append((source, target, row.to_dict()))
        for endpoint in (source, target):
            if endpoint not in node_seen:
                node_seen.add(endpoint)
                nodes.append(endpoint)
                orphan_endpoint_nodes.append(endpoint)

    if "text" in text_units_df.columns:
        source_text = "\n\n".join(text_units_df["text"].fillna("").astype(str))
    else:
        source_text = read_source_text(graph_dir)

    all_text_unit_ids = set(text_units_df.get("id", pd.Series(dtype=str)).fillna("").astype(str))
    all_text_unit_ids = {text_unit_id for text_unit_id in all_text_unit_ids if text_unit_id}
    entity_text_unit_ids = collect_text_unit_ids(entities_df)
    relationship_text_unit_ids = collect_text_unit_ids(relationships_df)
    covered_text_unit_ids = (entity_text_unit_ids | relationship_text_unit_ids) & all_text_unit_ids
    entity_covered_text_unit_ids = entity_text_unit_ids & all_text_unit_ids
    relationship_covered_text_unit_ids = relationship_text_unit_ids & all_text_unit_ids
    missing_text_unit_ids = sorted(all_text_unit_ids - covered_text_unit_ids)

    entity_types, entity_descriptions, edge_descriptions = parse_cache_records(graph_dir)
    for _idx, row in entities_df.iterrows():
        title = normalize_text(row.get("title", ""))
        if not title:
            continue
        entity_type = normalize_text(row.get("type", ""))
        description = normalize_text(row.get("description", ""))
        if entity_type:
            entity_types[normalize_key(title)][entity_type.upper()] += 1
        if description:
            entity_descriptions[normalize_key(title)].append(description)
    for _idx, row in relationships_df.iterrows():
        source = normalize_text(row.get("source", ""))
        target = normalize_text(row.get("target", ""))
        description = normalize_text(row.get("description", ""))
        if source and target and description:
            key = tuple(sorted((normalize_key(source), normalize_key(target))))
            edge_descriptions[key].append(description)
    all_descriptions = sum(entity_descriptions.values(), []) + sum(edge_descriptions.values(), [])
    graph_text = "\n".join(nodes + all_descriptions)

    node_type_counts = Counter(get_entity_type(node, entity_types) for node in nodes)
    relation_type_counts = Counter()
    for source, target, _data in edges:
        key = tuple(sorted((normalize_key(source), normalize_key(target))))
        relation_type_counts[classify_relation(source, target, edge_descriptions.get(key, []))] += 1

    graph_metrics = manual_graph_metrics(nodes, edges)
    components = graph_metrics["components"]
    largest_component = graph_metrics["largest_component"]
    bridges = graph_metrics["bridges"]
    degrees_of_freedom = graph_metrics["degrees_of_freedom"]
    cycles_count = graph_metrics["cycles_count"]
    degrees = graph_metrics["degrees"]

    noisy_nodes = [node for node in nodes if is_noisy(node)]
    unsupported_nodes = [node for node in nodes if not node_in_text(node, source_text)]
    unsupported_edges = [
        (source, target)
        for source, target, _data in edges
        if not edge_supported(source, target, source_text)
    ]
    coref_groups = detect_coreference_groups(nodes)

    text_norm = normalize_key(source_text)
    graph_norm = normalize_key(graph_text)
    terms_in_text = [term for term in DOMAIN_TERMS if normalize_key(term) in text_norm]
    terms_in_graph = [term for term in terms_in_text if normalize_key(term) in graph_norm]
    domain_coverage = len(terms_in_graph) / max(1, len(terms_in_text)) * 100
    primary_terms_in_text = [term for term in PRIMARY_DOMAIN_TERMS if normalize_key(term) in text_norm]
    primary_terms_in_graph = [term for term in primary_terms_in_text if normalize_key(term) in graph_norm]
    primary_terms_preservation = len(primary_terms_in_graph) / max(1, len(primary_terms_in_text)) * 100

    integrity = source_integrity(source_text, graph_text)
    # Полнота по тексту: доля чанков, на которые ссылаются вершины и/или ребра графа.
    completeness = len(covered_text_unit_ids) / max(1, len(all_text_unit_ids)) * 100
    relationship_chunk_coverage = len(relationship_covered_text_unit_ids) / max(1, len(all_text_unit_ids)) * 100
    consistency = (1 - len(unsupported_edges) / max(1, len(edges))) * 100
    noise_score = len(noisy_nodes) / max(1, len(nodes)) * 100

    return {
        "label": label,
        "path": str(graph_dir),
        "nodes": len(nodes),
        "edges": len(edges),
        "node_type_counts": dict(node_type_counts),
        "edge_type_counts": dict(relation_type_counts),
        "traversal_algorithm": graph_metrics["algorithm"],
        "text_completeness_pct": completeness,
        "total_text_units_count": len(all_text_unit_ids),
        "covered_text_units_count": len(covered_text_unit_ids),
        "missing_text_units_count": len(missing_text_unit_ids),
        "relationship_chunk_coverage_pct": relationship_chunk_coverage,
        "unsupported_nodes_count": len(unsupported_nodes),
        "consistency_pct": consistency,
        "unsupported_edges_count": len(unsupported_edges),
        "coreference_groups_count": len(coref_groups),
        "noisy_nodes_pct": noise_score,
        "noisy_nodes_count": len(noisy_nodes),
        "domain_terms_in_text": len(terms_in_text),
        "domain_terms_in_graph": len(terms_in_graph),
        "domain_coverage_pct": domain_coverage,
        "primary_terms_in_text": len(primary_terms_in_text),
        "primary_terms_in_graph": len(primary_terms_in_graph),
        "primary_terms_preservation_pct": primary_terms_preservation,
        "components": len(components),
        "largest_component_nodes": largest_component,
        "largest_component_pct": largest_component / max(1, len(nodes)) * 100,
        "bridges": len(bridges),
        "degrees_of_freedom": degrees_of_freedom,
        "cycles": cycles_count,
        "avg_degree": float(np.mean(list(degrees.values()))) if degrees else 0.0,
        "max_degree": max(degrees.values(), default=0),
        "integrity": integrity,
        "examples": {
            "unsupported_nodes": unsupported_nodes[:10],
            "missing_text_unit_ids": missing_text_unit_ids[:10],
            "unsupported_edges": unsupported_edges[:10],
            "coreference_groups": coref_groups[:10],
            "noisy_nodes": noisy_nodes[:10],
            "domain_terms_missing": [term for term in terms_in_text if term not in terms_in_graph][:10],
            "primary_terms_missing": [term for term in primary_terms_in_text if term not in primary_terms_in_graph][:10],
            "top_degree_nodes": sorted(degrees.items(), key=lambda item: item[1], reverse=True)[:10],
            "relationship_endpoints_missing_from_entities": orphan_endpoint_nodes[:10],
        },
    }


SUMMARY_METRICS = [
    "text_completeness_pct", "consistency_pct", "coreference_groups_count",
    "noisy_nodes_pct", "domain_coverage_pct", "primary_terms_preservation_pct", "nodes", "edges",
    "components", "largest_component_pct", "bridges", "degrees_of_freedom", "cycles",
]


def format_metric_value(value):
    if isinstance(value, (float, np.floating)):
        return f"{value:.2f}"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return str(value)


def build_comparison_table(results):
    rows = []
    for metric in SUMMARY_METRICS:
        row = {"metric": metric}
        for result in results:
            row[result["label"]] = format_metric_value(result[metric])
        rows.append(row)
    return pd.DataFrame(rows)


def format_graph_report(result):
    lines = []
    lines.append(f"{'=' * 24} {result['label']} {'=' * 24}")
    lines.append(f"Путь: {result['path']}")
    lines.append("")
    lines.append("Качество относительно текста")
    lines.append(
        f"  Полнота по тексту: {result['text_completeness_pct']:.2f}% "
        f"(покрыто чанков: {result['covered_text_units_count']} / {result['total_text_units_count']}, "
        f"не покрыто: {result['missing_text_units_count']})"
    )
    lines.append(
        f"  Непротиворечивость/поддержка ребер текстом: {result['consistency_pct']:.2f}% "
        f"(ребер без совместного упоминания: {result['unsupported_edges_count']})"
    )
    lines.append(f"  Кореференции: {result['coreference_groups_count']} групп возможных дублей")
    lines.append(f"  Шум в вершинах: {result['noisy_nodes_count']} ({result['noisy_nodes_pct']:.2f}%)")
    lines.append(
        f"  Целостность формул/таблиц: issues={result['integrity']['integrity_issues']} "
        f"(формул в тексте: {result['integrity']['source_formulas']}, "
        f"таблиц: {result['integrity']['source_tables']}, "
        f"картинок: {result['integrity']['source_images']})"
    )
    lines.append(
        f"  Покрытие ПрО: {result['domain_terms_in_graph']} / {result['domain_terms_in_text']} "
        f"({result['domain_coverage_pct']:.2f}%)"
    )
    lines.append(
        f"  Сохранность первых терминов до 16Г2АФ: "
        f"{result['primary_terms_in_graph']} / {result['primary_terms_in_text']} "
        f"({result['primary_terms_preservation_pct']:.2f}%)"
    )
    lines.append("")
    lines.append("Показатели графа")
    lines.append(f"  Алгоритм обхода: {result['traversal_algorithm']}")
    lines.append(f"  Вершины: {result['nodes']}")
    lines.append(f"  Ребра: {result['edges']}")
    lines.append(f"  Компоненты связности: {result['components']}")
    lines.append(
        f"  Крупнейшая компонента: {result['largest_component_nodes']} "
        f"({result['largest_component_pct']:.2f}%)"
    )
    lines.append(f"  Мосты: {result['bridges']}")
    lines.append(f"  Степени свободы / цикломатическое число: {result['degrees_of_freedom']}")
    lines.append(f"  Циклы: {result['cycles']}")
    lines.append(f"  Средняя степень: {result['avg_degree']:.2f}")
    lines.append(f"  Максимальная степень: {result['max_degree']}")
    lines.append("")
    lines.append("Вершины по типам")
    for key, value in sorted(result["node_type_counts"].items(), key=lambda item: (-item[1], item[0])):
        lines.append(f"  {key}: {value}")
    lines.append("")
    lines.append("Ребра по типам")
    for key, value in sorted(result["edge_type_counts"].items(), key=lambda item: (-item[1], item[0])):
        lines.append(f"  {key}: {value}")
    lines.append("")
    lines.append("Примеры для ручной проверки")
    for key, values in result["examples"].items():
        lines.append(f"  {key}: {values}")
    return "\n".join(lines)


def print_compact_graph_report(result):
    print(f"\n{'=' * 24} {result['label']} {'=' * 24}")
    print(f"Путь: {result['path']}")
    print(
        "Качество: "
        f"полнота={result['text_completeness_pct']:.2f}%, "
        f"непротиворечивость={result['consistency_pct']:.2f}%, "
        f"шум={result['noisy_nodes_pct']:.2f}%, "
        f"покрытие ПрО={result['domain_coverage_pct']:.2f}%, "
        f"первые термины={result['primary_terms_preservation_pct']:.2f}%"
    )
    print(
        "Граф: "
        f"вершины={result['nodes']}, "
        f"ребра={result['edges']}, "
        f"компоненты={result['components']}, "
        f"крупнейшая компонента={result['largest_component_pct']:.2f}%, "
        f"мосты={result['bridges']}, "
        f"циклы={result['cycles']}"
    )


def save_graph_reports(results, report_path="graph_comparison_report.txt", table_path="graph_comparison_summary.csv"):
    report_path = Path(report_path)
    table_path = Path(table_path)

    comparison_table = build_comparison_table(results)
    full_report = "\n\n".join(format_graph_report(result) for result in results)

    if len(results) >= 2:
        full_report += "\n\n" + "=" * 24 + " Сравнение графов " + "=" * 24
        full_report += "\n" + comparison_table.to_string(index=False)

    report_path.write_text(full_report, encoding="utf-8")
    comparison_table.to_csv(table_path, index=False, encoding="utf-8-sig")
    return report_path.resolve(), table_path.resolve(), comparison_table


graph_quality_results = [analyze_graph(label, path) for label, path in GRAPH_DIRS.items()]

for result in graph_quality_results:
    print_compact_graph_report(result)

report_path, table_path, comparison_table = save_graph_reports(graph_quality_results)

if len(graph_quality_results) >= 2:
    print("\n" + "=" * 24 + " Сравнение графов " + "=" * 24)
    print(comparison_table.to_string(index=False))

print(f"\nПолный отчет сохранен: {report_path}")
print(f"CSV-таблица сохранена: {table_path}")
